# Local series

The first cell moves to the repository root, so the notebook works whether Jupyter starts in `notebooks/` or at the root.

The plot cell loads `data/series.npy` when that file exists. The array should be numeric, with one or two dimensions. The figure is saved as `outputs/series.png`. `data/` and `outputs/` are gitignored, and so are `*.npy` files.

In [ ]:
%load_ext autoreload
%autoreload 2

In [ ]:
import os
from pathlib import Path

# Jupyter starts in notebooks/. Move to the repository root (the folder that
# contains pyproject.toml) so conf/, data/ and outputs/ resolve.
root = Path.cwd()
while not (root / "pyproject.toml").is_file() and root != root.parent:
    root = root.parent
os.chdir(root)
print(f"Working directory: {root}")

In [ ]:
from pathlib import Path

import matplotlib.pyplot as plt
import numpy as np

series_path = Path("data/series.npy")
figure_path = Path("outputs/series.png")

if not series_path.is_file():
    print(
        "data/series.npy is missing. data/ is gitignored, and so are *.npy files, "
        "so the series is not in git. From the repository root, create data/ and "
        "save a 1D or 2D numeric array as data/series.npy, then run this cell again. "
        "Importing the library does not need that file."
    )
else:
    series = np.load(series_path)
    numeric = isinstance(series, np.ndarray) and np.issubdtype(series.dtype, np.number)
    if not numeric or series.ndim not in (1, 2):
        print(
            f"{series_path.as_posix()} must be a 1D or 2D numeric array. "
            "Replace the file and run this cell again."
        )
    else:
        figure_path.parent.mkdir(parents=True, exist_ok=True)
        fig, ax = plt.subplots()
        ax.plot(series)
        ax.set_xlabel("index")
        ax.set_ylabel("value")
        ax.set_title(series_path.as_posix())
        fig.savefig(figure_path)
        plt.show()
        print(f"Wrote {figure_path.as_posix()}. outputs/ is gitignored.")

## Library

The next cell composes `conf/` the same way as the tests and builds the transverse-field Ising model selected by `sim_type=TFIM`. `model.evolve` takes either a state vector `psi` of shape `(dim,)` or a density matrix `rho` of shape `(dim, dim)` and returns the evolved object of the same kind. The cell evolves `|0...0>` both ways for one step `dt` and checks that they agree.

In [ ]:
import numpy as np
from hydra import compose, initialize_config_dir
from hydra.core.global_hydra import GlobalHydra

from QELM import TransverseFieldIsingModel, set_seed

GlobalHydra.instance().clear()
with initialize_config_dir(version_base="1.3", config_dir=str(Path("conf").resolve())):
    cfg = compose(config_name="config", overrides=["sim_type=TFIM", "TFIM.N=6"])
set_seed(cfg)
model = TransverseFieldIsingModel.from_config(cfg)
print(f"N={model.N} J={model.J} h={model.h} dim={model.dim}")
dt = float(cfg.test.dt)
psi0 = np.zeros(model.dim, dtype=complex)
psi0[0] = 1.0
psi = model.evolve(dt=dt, state=psi0)
rho = model.evolve(dt=dt, state=np.outer(psi0, psi0.conj()))
print(f"psi: shape={psi.shape} norm={np.linalg.norm(psi):.12f}")
print(f"rho: shape={rho.shape} trace={np.trace(rho).real:.12f}")
print(f"rho == |psi><psi|: {np.allclose(rho, np.outer(psi, psi.conj()))}")